# Neural Network Training with TensorFlow, PyTorch and Logistic Regression

This notebook demonstrates:
1. **Neural Network Training with TensorFlow** - Using Keras on MNIST dataset
2. **Neural Network Training with PyTorch** - Custom neural network with backpropagation
3. **Logistic Regression with TensorFlow** - Binary classification evaluation

All methods show how to import libraries, build models, train, and evaluate performance.

In [ ]:
# 1. Neural Network Training with TensorFlow on MNIST Dataset

import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

# Load MNIST dataset
print("Loading MNIST dataset...")
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize data to [0, 1]
x_train, x_test = x_train / 255.0, x_test / 255.0

# Build Sequential model
print("\nBuilding TensorFlow model...")
model_tf = models.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

# Compile model
model_tf.compile(optimizer='adam',
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])

# Train model
print("Training TensorFlow model...")
history = model_tf.fit(x_train, y_train, epochs=5, batch_size=32, verbose=1)

# Evaluate on test data
print("\nEvaluating on test data...")
test_loss, test_accuracy = model_tf.evaluate(x_test, y_test, verbose=0)
print(f"TensorFlow Model - Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

In [ ]:
# 2. Neural Network Training with PyTorch

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Define Neural Network
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(784, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Initialize model, loss, and optimizer
print("\nBuilding PyTorch model...")
model_pytorch = Net().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_pytorch.parameters(), lr=0.001)

# Prepare data
print("Preparing data...")
x_train_tensor = torch.from_numpy(x_train).float().to(device)
y_train_tensor = torch.from_numpy(y_train).long().to(device)
x_test_tensor = torch.from_numpy(x_test).float().to(device)
y_test_tensor = torch.from_numpy(y_test).long().to(device)

train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Training loop
print("Training PyTorch model...")
num_epochs = 5
for epoch in range(num_epochs):
    total_loss = 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model_pytorch(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")

# Evaluate on test data
print("\nEvaluating on test data...")
model_pytorch.eval()
with torch.no_grad():
    test_outputs = model_pytorch(x_test_tensor)
    _, predicted = torch.max(test_outputs, 1)
    accuracy = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
    print(f"PyTorch Model - Test Accuracy: {accuracy:.4f}")

In [ ]:
# 3. Logistic Regression using TensorFlow (Binary Classification)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# Generate binary classification dataset
print("Generating binary classification dataset...")
X, y = make_classification(n_samples=1000, n_features=20, n_informative=15, 
                           n_redundant=5, n_classes=2, random_state=42)

# Split into train and test
X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(X, y, test_size=0.2, random_state=42)

# Build Logistic Regression model (single neuron with sigmoid)
print("\nBuilding Logistic Regression model...")
model_lr = Sequential([
    Dense(1, activation='sigmoid', input_shape=(20,))
])

# Compile with binary crossentropy loss
model_lr.compile(optimizer='adam',
                 loss='binary_crossentropy',
                 metrics=['accuracy'])

# Train the model
print("Training Logistic Regression model...")
history_lr = model_lr.fit(X_train_lr, y_train_lr, epochs=20, batch_size=32, verbose=1)

# Evaluate on test data
print("\nEvaluating on test data...")
loss, accuracy = model_lr.evaluate(X_test_lr, y_test_lr, verbose=0)
print(f"Logistic Regression - Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}")

# Make predictions
predictions = model_lr.predict(X_test_lr[:5], verbose=0)
print("\nSample Predictions (first 5 test samples):")
for i, pred in enumerate(predictions):
    print(f"Sample {i+1}: {pred[0]:.4f} (Actual: {y_test_lr[i]})")